# Tuning


In [1]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

import joblib
import os
import pandas as pd
import numpy as np
import optuna

from scipy import stats

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, \
StratifiedShuffleSplit, GridSearchCV, RandomizedSearchCV

from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, ElasticNet,\
RidgeClassifier, LogisticRegressionCV

from lightgbm import LGBMClassifier, LGBMRegressor, early_stopping
from catboost import CatBoostClassifier, CatBoostRegressor, Pool

from optuna.integration import CatBoostPruningCallback

from sklearn.metrics import mean_absolute_error, mean_squared_error, f1_score, accuracy_score, \
roc_auc_score, classification_report, r2_score, precision_score, recall_score, \
log_loss, mean_squared_log_error

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder

import matplotlib.pyplot as plt
import seaborn as sns
            
from tqdm import tqdm_notebook
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

import ipywidgets
from get_metrics import get_metrics_classification

RAND = 10
N_FOLDS = 5

In [2]:
# Загрузка моделей и датасетов

df_train =  pd.read_csv("df_train_lr.csv")
df_train_2 = pd.read_csv("df_train_2.csv")

model_Sex_F = joblib.load('model_Sex_F.pkl') #LR Sex_F
model_ADHD = joblib.load('model_ADHD.pkl') #LR ADHD
clf_1 = joblib.load('clf_1.pkl') #LightGBM ADHD
clf_2 = joblib.load('clf_2.pkl') #LightGBM Sex_F
clf_c_1 = joblib.load('clf_c_1.pkl') #CatBoost ADHD
clf_c_2 = joblib.load('clf_c_2.pkl') #CatBoost Sex_F

In [3]:
# Разделение для логистической регрессии 
X = df_train.drop(['ADHD_Outcome', 'Sex_F'], axis=1)
y_ADHD = df_train['ADHD_Outcome'].values
y_Sex_F = df_train['Sex_F'].values

X_train_1_ADHD, X_test_1_ADHD, y_train_1_ADHD, y_test_1_ADHD = train_test_split(X,
                                                    y_ADHD,
                                                    test_size=0.2,
                                                    random_state=RAND,
                                                    stratify=y_ADHD)

X_train_1_ADHD_, X_val_1_ADHD, y_train_1_ADHD_, y_val_1_ADHD = train_test_split(X_train_1_ADHD,
                                                    y_train_1_ADHD,
                                                    test_size=0.16,
                                                    shuffle=True,
                                                    random_state=RAND)

X_train_1_Sex_F, X_test_1_Sex_F, y_train_1_Sex_F, y_test_1_Sex_F = train_test_split(X,
                                                    y_Sex_F,
                                                    test_size=0.2,
                                                    random_state=RAND,
                                                    stratify=y_Sex_F)


X_train_1_Sex_F_, X_val_1_Sex_F, y_train_1_Sex_F_, y_val_1_Sex_F = train_test_split(X_train_1_Sex_F,
                                                    y_train_1_Sex_F,
                                                    test_size=0.16,
                                                    shuffle=True,
                                                    random_state=RAND)

In [4]:
# Разделение для бустингов
X_cat = df_train_2.drop(['ADHD_Outcome', 'Sex_F'], axis=1)
y_cat_ADHD = df_train_2['ADHD_Outcome']
y_cat_Sex_F = df_train_2['Sex_F']

X_train_ADHD, X_test_ADHD, y_train_ADHD, y_test_ADHD = train_test_split(X_cat,
                                                    y_cat_ADHD,
                                                    test_size=0.2,
                                                    shuffle=True,
                                                    random_state=RAND)

X_train_ADHD_, X_val_ADHD, y_train_ADHD_, y_val_ADHD = train_test_split(X_train_ADHD,
                                                    y_train_ADHD,
                                                    test_size=0.16,
                                                    shuffle=True,
                                                    random_state=RAND)

X_train_Sex_F, X_test_Sex_F, y_train_Sex_F, y_test_Sex_F = train_test_split(X_cat,
                                                    y_cat_Sex_F,
                                                    test_size=0.2,
                                                    shuffle=True,
                                                    random_state=RAND)

X_train_Sex_F_, X_val_Sex_F, y_train_Sex_F_, y_val_Sex_F = train_test_split(X_train_Sex_F,
                                                    y_train_Sex_F,
                                                    test_size=0.16,
                                                    shuffle=True,
                                                    random_state=RAND)

In [5]:
# Предсказывание для лоогистической регрессии
y_prob_ADHD_train = model_ADHD.predict_proba(X_val_1_ADHD)
y_pred_ADHD_train = model_ADHD.predict(X_val_1_ADHD)

y_prob_Sex_F_train = model_Sex_F.predict_proba(X_val_1_Sex_F)
y_pred_Sex_F_train = model_Sex_F.predict(X_val_1_Sex_F)

# Предсказывание для LightGBM
y_pred_1 = clf_1.predict(X_val_ADHD)
y_pred_2 = clf_2.predict(X_val_Sex_F)

# Предсказывание CatBoost
y_pred_c_1 = clf_c_1.predict(X_val_ADHD)
y_pred_c_2 = clf_c_2.predict(X_val_Sex_F)

  File "C:\Users\User\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 199, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\User\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\User\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\User\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


In [6]:
#Выведем метрики на Baseline для ADHD
metrics_ADHD = get_metrics_classification(y_val_1_ADHD,
                           y_pred_ADHD_train,
                           y_prob_ADHD_train,
                           name='LogisticRegression_Baseline_ADHD')
metrics_ADHD = pd.concat([metrics_ADHD,
    get_metrics_classification(y_test=y_val_ADHD,
                               y_pred=y_pred_1,
                               y_score=clf_1.predict_proba(X_val_ADHD),
                               name='LightGBM_eval_Baseline_ADHD')])
metrics_ADHD = pd.concat([metrics_ADHD,
    get_metrics_classification(y_test=y_val_ADHD,
                               y_pred=y_pred_c_1,
                               y_score=clf_c_1.predict_proba(X_val_ADHD),
                               name='CatBoost_eval_Baseline_ADHD')])

round(metrics_ADHD.set_index('model'), 3)

,Accuracy,ROC_AUC,Precision,Recall,f1,Logloss
model,,,,,,
LogisticRegression_Baseline_ADHD,0.636,0.585,0.727,0.762,0.744,1.400
LightGBM_eval_Baseline_ADHD,0.702,0.788,0.698,0.926,0.796,0.565
CatBoost_eval_Baseline_ADHD,0.689,0.743,0.682,0.947,0.793,0.633


In [7]:
#Выведем метрики на Baseline для Sex_F
metrics_Sex_F = get_metrics_classification(y_val_1_Sex_F,
                                     y_pred_Sex_F_train,
                                     y_prob_Sex_F_train,
                                     name='LogisticRegression_Baseline_Sex_F')
metrics_Sex_F = pd.concat([metrics_Sex_F,
    get_metrics_classification(y_test=y_val_Sex_F,
                               y_pred=y_pred_2,
                               y_score=clf_2.predict_proba(X_val_Sex_F),
                               name='LightGBM_Baseline_Sex_F')])
metrics_Sex_F = pd.concat([metrics_Sex_F,
    get_metrics_classification(y_test=y_val_Sex_F,
                               y_pred=y_pred_c_2,
                               y_score=clf_c_2.predict_proba(X_val_Sex_F),
                               name='CatBoost_Baseline_Sex_F')])

round(metrics_Sex_F.set_index('model'), 3)

,Accuracy,ROC_AUC,Precision,Recall,f1,Logloss
model,,,,,,
LogisticRegression_Baseline_Sex_F,0.709,0.724,0.541,0.426,0.476,0.990
LightGBM_Baseline_Sex_F,0.669,0.723,0.625,0.185,0.286,0.643
CatBoost_Baseline_Sex_F,0.675,0.756,0.778,0.130,0.222,0.573


# Optuna


In [8]:
def objective_lr(trial, X, y, N_FOLDS, random_state):
    #Гиперпараметры для подбора
    lr_params = {
        "C": trial.suggest_float('C', 1e-4, 10.0, log=True),
        "penalty": trial.suggest_categorical('penalty', ['l1', 'l2']),
        "solver": trial.suggest_categorical('solver', ['liblinear', 'saga'])
        } 
    #Кросс-валидация
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=random_state)
    cv_predicts = np.empty(N_FOLDS)

    for idx, (train_idx, test_idx) in enumerate(cv.split(X, y)):

        X_train_, X_val = X.iloc[train_idx], X.iloc[test_idx]
        y_train_, y_val = y[train_idx], y[test_idx]
        
        # Модель LogisticRegression    
        model = LogisticRegression(**lr_params)
        model.fit(X_train_,
                  y_train_,
                 )  
        preds = model.predict(X_val)
        cv_predicts[idx] = roc_auc_score(y_val, preds)

    return np.mean(cv_predicts)

In [9]:
#Сравнение по метрике ROC-AUC модель LR baseline и optune 
def evaluate_cv_lr(model, X, y, random_state, N_FOLDS = 5):

    cv_scores = []

    cv = StratifiedKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=random_state
    )
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):

        X_train_ = X.iloc[train_idx]
        X_val_ = X.iloc[val_idx]

        y_train_ = y[train_idx]
        y_val_ = y[val_idx]

        model.fit(
            X_train_,
            y_train_,
        )

        y_pred_proba = model.predict_proba(X_val_)[:, 1]

        score = roc_auc_score(y_val_, y_pred_proba)
        cv_scores.append(score)
        print(f"Fold {fold}: ROC-AUC:{score:.4f}")
    print(f"\nMean ROC-AUC: {np.mean(cv_scores):.4f}")
    print(f"Std: {np.std(cv_scores):.4f}")

    return np.mean(cv_scores), np.std(cv_scores)

In [10]:
def objective_lgb(trial, X, y, N_FOLDS, random_state):
    #Гиперпараметры для подбора
    lgb_params = {
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "max_bin": trial.suggest_int("max_bin", 128, 255),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 2000),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.8, 0.9),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.01, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 1.0, log=True),
        "random_state": RAND,
        "n_jobs": None,
        "verbosity": -1
    }

    #Кросс-валидация
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=random_state)
    cv_predicts = []

    for idx, (train_idx, test_idx) in enumerate(cv.split(X, y)):
        X_train_, X_val = X.iloc[train_idx], X.iloc[test_idx]
        y_train_, y_val = y.iloc[train_idx], y.iloc[test_idx]

        # Модель LightGBM
        model = LGBMClassifier(**lgb_params)
        pruning_callback = optuna.integration.LightGBMPruningCallback(
            trial, "auc")

        model.fit(
            X_train_,
            y_train_,
            eval_set=[(X_val, y_val)],
            eval_metric="auc",
            callbacks=[
                       early_stopping(stopping_rounds=100)]
        )

        y_pred_proba = model.predict_proba(X_val)[:, 1]
        cv_predict = roc_auc_score(y_val, y_pred_proba)
        cv_predicts.append(cv_predict)

    return np.mean(cv_predicts)

In [11]:
#Сравнение по метрике ROC-AUC модель LightGBM baseline и optune 
def evaluate_cv_lgb(model, X, y, random_state, N_FOLDS = 5):

    cv_scores = []

    cv = StratifiedKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=random_state
    )
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):

        X_train_ = X.iloc[train_idx]
        X_val_ = X.iloc[val_idx]

        y_train_ = y.iloc[train_idx]
        y_val_ = y.iloc[val_idx]

        model.fit(
            X_train_,
            y_train_,
            eval_set=[(X_val_, y_val_)],
            eval_metric="auc",
            callbacks=[
                       early_stopping(stopping_rounds=100)
                       ]
        )
        
        y_pred_proba = model.predict_proba(X_val_)[:, 1]

        score = roc_auc_score(y_val_, y_pred_proba)
        cv_scores.append(score)
        print(f"Fold {fold}: ROC-AUC:{score:.4f}")
    print(f"\nMean ROC-AUC: {np.mean(cv_scores):.4f}")
    print(f"Std: {np.std(cv_scores):.4f}")

    return np.mean(cv_scores), np.std(cv_scores)

In [12]:
def objective_catboost(trial, X, y, N_FOLDS, random_state):
    #Гиперпараметры для подбора
    cb_params = {
        "iterations": trial.suggest_int("iterations", 100, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "depth": trial.suggest_int("depth", 4, 6),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "border_count": trial.suggest_int("border_count", 32, 128),
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_state": random_state,
        "task_type": "CPU",
        "verbose": 100,
        "allow_writing_files": False,
        "od_type": "Iter",
        "od_wait": 30,  # ранняя остановка
    }
    #Кросс-валидация
    cv_scores = []

    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=random_state)
    
    for train_idx, val_idx in cv.split(X, y):

        X_train_, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train_, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # Модель CatBoostClassifier         
        model = CatBoostClassifier(**cb_params)
        model.fit(
            X_train_, y_train_,
            eval_set = (X_val, y_val),
            use_best_model = True,
            early_stopping_rounds=100
        )
        
        y_pred_proba = model.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, y_pred_proba)
        cv_scores.append(score)

    return np.mean(cv_scores)

In [13]:
#Сравнение по метрике ROC-AUC модель CatBoost baseline и optune 
def evaluate_cv_catboost(model, X, y, random_state, N_FOLDS = 5):

    cv_scores = []

    cv = StratifiedKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=random_state
    )
    X = pd.DataFrame(X)
    y = pd.DataFrame(y)
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):

        X_train_ = X.iloc[train_idx]
        X_val_ = X.iloc[val_idx]

        y_train_ = y.iloc[train_idx]
        y_val_ = y.iloc[val_idx]

        model.fit(
            X_train_,
            y_train_,
            eval_set=(X_val_, y_val_),
            use_best_model=True,
            early_stopping_rounds=100,
            verbose = False
        )

        y_pred_proba = model.predict_proba(X_val_)[:, 1]

        score = roc_auc_score(y_val_, y_pred_proba)
        cv_scores.append(score)
        print(f"Fold {fold}: ROC-AUC:{score:.4f}")
    print(f"\nMean ROC-AUC: {np.mean(cv_scores):.4f}")
    print(f"Std: {np.std(cv_scores):.4f}")

    return np.mean(cv_scores), np.std(cv_scores)

In [14]:
#Нахождение гиперпараметров LR для таргета ADHD

study = optuna.create_study(direction="maximize", study_name="LR_ADHD")
func = lambda trial: objective_lr(
    trial, X_train_1_ADHD_, y_train_1_ADHD_, N_FOLDS=N_FOLDS, random_state=RAND)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(func, n_trials=30, show_progress_bar=True)

study.best_params

In [15]:
#Лучшие параметры для LR ADHD 
best_params_lr_ADHD = {'C': 0.00031170923197518425, 'penalty': 'l2', 'solver': 'liblinear'}  #ROC_AUC 0.578468	

In [16]:
clf_ADHD_lr = LogisticRegression(**best_params_lr_ADHD)

clf_ADHD_lr.fit(X_train_1_ADHD_,
        y_train_1_ADHD_
        )

y_pred_ADHD_lr = clf_ADHD_lr.predict(X_val_1_ADHD)

In [17]:
metrics_optuna_ADHD = get_metrics_classification(y_test=y_val_1_ADHD,
                               y_pred=y_pred_ADHD_lr,
                               y_score=clf_ADHD_lr.predict_proba(X_val_1_ADHD),
                               name='LogisticRegression_Optuna_ADHD')

metrics_optuna_ADHD

,model,Accuracy,ROC_AUC,Precision,Recall,f1,Logloss
0,LogisticRegression_Optuna_ADHD,0.483444,0.578468,0.721311,0.419048,0.53012,0.739364


In [18]:
metrics_ADHD

,model,Accuracy,ROC_AUC,Precision,Recall,f1,Logloss
0,LogisticRegression_Baseline_ADHD,0.635762,0.584679,0.727273,0.761905,0.744186,1.399807
0,LightGBM_eval_Baseline_ADHD,0.701987,0.787970,0.698413,0.926316,0.796380,0.564567
0,CatBoost_eval_Baseline_ADHD,0.688742,0.743233,0.681818,0.947368,0.792952,0.633315


In [19]:
#На валидационной выборке модель полученная с помощью Optune проигрывает по метрикам, для точности проверю на 5 фолдах

In [20]:
#Значения на 5 фолдах для baseline LR для ADHD
ADHD_base_lr_mean, ADHD_base_lr_std = evaluate_cv_lr(
    model_ADHD,
    X_train_1_ADHD_,
    y_train_1_ADHD_,
    RAND)

Fold 1: ROC-AUC:0.5800
Fold 2: ROC-AUC:0.5804
Fold 3: ROC-AUC:0.5083
Fold 4: ROC-AUC:0.5687
Fold 5: ROC-AUC:0.6056

Mean ROC-AUC: 0.5686
Std: 0.0325


In [21]:
#Значения на 5 фолдах для opune LR для ADHD
ADHD_optune_lr_mean, ADHD_optune_lr_std = evaluate_cv_lr(
    clf_ADHD_lr,
    X_train_1_ADHD_,
    y_train_1_ADHD_,
    RAND)

Fold 1: ROC-AUC:0.5752
Fold 2: ROC-AUC:0.6034
Fold 3: ROC-AUC:0.5317
Fold 4: ROC-AUC:0.5607
Fold 5: ROC-AUC:0.6061

Mean ROC-AUC: 0.5754
Std: 0.0278


In [22]:
cv_models_ADHD = []
cv_models_ADHD.append({'Model_ADHD':'LogisticRegression_baseline_ADHD',
                               'Mean ROC-AUC':ADHD_base_lr_mean,
                               'Std': ADHD_base_lr_std})
cv_models_ADHD.append({'Model_ADHD':'LogisticRegression_optune_ADHD',
                               'Mean ROC-AUC':ADHD_optune_lr_mean,
                               'Std':ADHD_optune_lr_std})
cv_models_ADHD_df = pd.DataFrame(cv_models_ADHD)
round(cv_models_ADHD_df.set_index('Model_ADHD'), 3)

,Mean ROC-AUC,Std
Model_ADHD,,
LogisticRegression_baseline_ADHD,0.569,0.032
LogisticRegression_optune_ADHD,0.575,0.028


In [23]:
#На 5 фолдах уже можно увидеть, что метрика ROC-AUC модели LR для ADHD полученная через Optune выше на 0.5754 - 0.5686 = 0.0068

study_lr_F = optuna.create_study(direction="maximize", study_name="LR_SF")
func = lambda trial: objective_lr(
    trial, X_train_1_Sex_F_, y_train_1_Sex_F_, N_FOLDS=N_FOLDS, random_state=RAND)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_lr_F.optimize(func, n_trials=30, show_progress_bar=True)

study_lr_F.best_params

In [24]:
#Лучшие параметры для LR для Sex_F на 04.09
best_params_lr_Sex_f = {'C': 0.0004360436996760946, 'penalty': 'l2', 'solver': 'liblinear'}

In [25]:
clf_Sex_F_lr = LogisticRegression(**best_params_lr_Sex_f)

clf_Sex_F_lr.fit(X_train_Sex_F_,
        y_train_Sex_F_,
        )

y_pred_Sex_F_lr = clf_Sex_F_lr.predict(X_val_Sex_F)

In [26]:
metrics_optuna_Sex_F = get_metrics_classification(y_test=y_val_Sex_F,
                               y_pred=y_pred_Sex_F_lr,
                               y_score=clf_Sex_F_lr.predict_proba(X_val_Sex_F),
                               name='LogisticRegression_Optuna_Sex_F')

metrics_optuna_Sex_F

,model,Accuracy,ROC_AUC,Precision,Recall,f1,Logloss
0,LogisticRegression_Optuna_Sex_F,0.642384,0.724895,0.0,0.0,0.0,0.616547


In [27]:
metrics_Sex_F

,model,Accuracy,ROC_AUC,Precision,Recall,f1,Logloss
0,LogisticRegression_Baseline_Sex_F,0.708609,0.723813,0.540541,0.425532,0.476190,0.990092
0,LightGBM_Baseline_Sex_F,0.668874,0.723368,0.625000,0.185185,0.285714,0.642981
0,CatBoost_Baseline_Sex_F,0.675497,0.756396,0.777778,0.129630,0.222222,0.572879


In [28]:
#Значения на 5 фолдах для baseline LR для Sex_F
Sex_F_base_lr_mean, Sex_F_base_lr_std = evaluate_cv_lr(
    model_Sex_F,
    X_train_1_Sex_F_,
    y_train_1_Sex_F_,
    RAND)

Fold 1: ROC-AUC:0.7852
Fold 2: ROC-AUC:0.6955
Fold 3: ROC-AUC:0.7908
Fold 4: ROC-AUC:0.7575
Fold 5: ROC-AUC:0.7092

Mean ROC-AUC: 0.7476
Std: 0.0389


In [29]:
#Значения на 5 фолдах для optune LR для Sex_F
Sex_F_optune_lr_mean, Sex_F_optune_lr_std = evaluate_cv_lr(
    clf_Sex_F_lr,
    X_train_1_Sex_F_,
    y_train_1_Sex_F_,
    RAND)

Fold 1: ROC-AUC:0.7711
Fold 2: ROC-AUC:0.7013
Fold 3: ROC-AUC:0.7751
Fold 4: ROC-AUC:0.7573
Fold 5: ROC-AUC:0.7133

Mean ROC-AUC: 0.7436
Std: 0.0305


In [30]:
cv_models_Sex_F = []
cv_models_Sex_F.append({'Model_Sex_F':'LogisticRegression_baseline_Sex_F',
                               'Mean ROC-AUC':Sex_F_base_lr_mean,
                               'Std': Sex_F_base_lr_std})
cv_models_Sex_F.append({'Model_Sex_F':'LogisticRegression_optune_Sex_F',
                               'Mean ROC-AUC':Sex_F_optune_lr_mean,
                               'Std':Sex_F_optune_lr_std})
cv_models_Sex_F_df = pd.DataFrame(cv_models_Sex_F)
round(cv_models_Sex_F_df.set_index('Model_Sex_F'), 3)

,Mean ROC-AUC,Std
Model_Sex_F,,
LogisticRegression_baseline_Sex_F,0.748,0.039
LogisticRegression_optune_Sex_F,0.744,0.030


In [31]:
#Для таргета Sex_F логистическая регрессия на бейзлайне сработала лучше по метрике ROC-AUC на 0.004

study_lg_A = optuna.create_study(direction="maximize", study_name="LGB_ADHD")
func = lambda trial: objective_lgb(
    trial, X_train_ADHD_, y_train_ADHD_, N_FOLDS=N_FOLDS, random_state=RAND)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_lg_A.optimize(func, n_trials=30, show_progress_bar=True)

study_lg_A.best_params

In [32]:
#лучшие параметры для ADHD для LGBM #ROC AUC 0.733 на пяти фолдах 0.6925 0.7533
best_params_ADHD_lgb = {'num_leaves': 27,
 'max_depth': 6,
 'max_bin': 142,
 'learning_rate': 0.015367920082427804,
 'n_estimators': 1807,
 'min_child_samples': 15,
 'subsample': 0.8446879181304909,
 'reg_alpha': 0.022770476384198494,
 'reg_lambda': 0.121697317902343}

In [33]:
clf_ADHD_lgb = LGBMClassifier(**best_params_ADHD_lgb, verbosity = -1)
eval_set = [(X_val_ADHD, y_val_ADHD)]

clf_ADHD_lgb.fit(X_train_ADHD_,
        y_train_ADHD_,
        eval_set=eval_set,
        eval_metric ='auc'
        )

y_pred_ADHD_lgb = clf_ADHD_lgb.predict(X_val_ADHD)

In [34]:
metrics_optuna_ADHD = pd.concat([metrics_optuna_ADHD,
    get_metrics_classification(y_test=y_val_ADHD,
                               y_pred=y_pred_ADHD_lgb,
                               y_score=clf_ADHD_lgb.predict_proba(X_val_ADHD),
                               name='LightGBM_Optuna_ADHD')])

round(metrics_optuna_ADHD.set_index('model'), 3)

,Accuracy,ROC_AUC,Precision,Recall,f1,Logloss
model,,,,,,
LogisticRegression_Optuna_ADHD,0.483,0.578,0.721,0.419,0.53,0.739
LightGBM_Optuna_ADHD,0.689,0.733,0.671,0.989,0.80,0.991


In [35]:
#Проверим на 5 фолдах LightGBM и сравним baseline и optune для ADHD

In [36]:
#Значения на 5 фолдах для baseline LightGBM для ADHD
ADHD_base_lgbm_mean, ADHD_base_lgbm_std = evaluate_cv_lgb(
    clf_1,
    X_train_ADHD_,
    y_train_ADHD_,
    RAND)

Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[21]	valid_0's auc: 0.739205
Fold 1: ROC-AUC:0.7392
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[7]	valid_0's auc: 0.733333
Fold 2: ROC-AUC:0.7333
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[27]	valid_0's auc: 0.710606
Fold 3: ROC-AUC:0.7106
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[88]	valid_0's auc: 0.764015
Fold 4: ROC-AUC:0.7640
Training until validation scores don't improve for 100 rounds
Training until validation scores don't impr

In [37]:
#Значения на 5 фолдах для opune LightGBM для ADHD
ADHD_optune_lgbm_mean, ADHD_optune_lgbm_std = evaluate_cv_lgb(
    clf_ADHD_lgb,
    X_train_ADHD_,
    y_train_ADHD_,
    RAND)

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[102]	valid_0's auc: 0.803977	valid_0's binary_logloss: 0.524128
Fold 1: ROC-AUC:0.8040
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[145]	valid_0's auc: 0.706439	valid_0's binary_logloss: 0.570153
Fold 2: ROC-AUC:0.7064
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[106]	valid_0's auc: 0.792235	valid_0's binary_logloss: 0.523338
Fold 3: ROC-AUC:0.7922
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[204]	valid_0's auc: 0.757197	valid_0's binary_logloss: 0.52833
Fold 4: ROC-AUC:0.7572
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[119]	valid_0's auc: 0.706422	valid_0's binary_logloss: 0.558191
Fold 5: ROC-AUC:0.7064

Mean ROC-AUC: 0.7533
Std: 0.0412


In [38]:
cv_models_ADHD.append({'Model_ADHD':'LightGBM_baseline_ADHD',
                               'Mean ROC-AUC':ADHD_base_lgbm_mean,
                               'Std': ADHD_base_lgbm_std})
cv_models_ADHD.append({'Model_ADHD':'LightGBM_optune_ADHD',
                               'Mean ROC-AUC':ADHD_optune_lgbm_mean,
                               'Std':ADHD_optune_lgbm_std})
cv_models_ADHD_df = pd.DataFrame(cv_models_ADHD)
round(cv_models_ADHD_df.set_index('Model_ADHD'), 3)

,Mean ROC-AUC,Std
Model_ADHD,,
LogisticRegression_baseline_ADHD,0.569,0.032
LogisticRegression_optune_ADHD,0.575,0.028
LightGBM_baseline_ADHD,0.740,0.018
LightGBM_optune_ADHD,0.753,0.041


In [39]:
metrics_optuna_ADHD = pd.concat([metrics_optuna_ADHD,
    get_metrics_classification(y_test=y_val_ADHD,
                               y_pred=y_pred_ADHD_lgb,
                               y_score=clf_ADHD_lgb.predict_proba(X_val_ADHD),
                               name='LightGBM_Optuna_ADHD')])

round(metrics_optuna_ADHD.set_index('model'), 3)

,Accuracy,ROC_AUC,Precision,Recall,f1,Logloss
model,,,,,,
LogisticRegression_Optuna_ADHD,0.483,0.578,0.721,0.419,0.53,0.739
LightGBM_Optuna_ADHD,0.689,0.733,0.671,0.989,0.80,0.991
LightGBM_Optuna_ADHD,0.689,0.693,0.671,0.989,0.80,0.616


study_lg_S = optuna.create_study(direction="maximize", study_name="LGB_SF")
func = lambda trial: objective_lgb(
    trial, X_train_Sex_F_, y_train_Sex_F_, N_FOLDS=N_FOLDS, random_state=RAND)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_lg_S.optimize(func, n_trials=30, show_progress_bar=True)

study_lg_S.best_params

In [40]:
best_params_Sex_F_lgbm = {'boosting_type': 'gbdt',  #ROC_AUC 0.731195 на пяти фолдах 0.7051
 'num_leaves': 31,
 'max_depth': 7,
 'max_bin': 155,
 'learning_rate': 0.024226661799654204,
 'n_estimators': 1100,
 'min_child_samples': 80,
 'min_split_gain': 6.902201580309832,
 'subsample': 0.6168798865047278,
 'subsample_freq': 1,
 'colsample_bytree': 0.6141516293976439,
 'reg_alpha': 0.000628329986467489,
 'reg_lambda': 7.142504420569093e-08}

In [41]:
clf_Sex_F_lgb = LGBMClassifier(**best_params_Sex_F_lgbm, verbosity = -1)
eval_set = [(X_val_Sex_F, y_val_Sex_F)]

clf_Sex_F_lgb.fit(X_train_Sex_F_,
        y_train_Sex_F_,
        eval_set=eval_set,
        eval_metric ='auc'
        )

y_pred_Sex_F_lgb = clf_Sex_F_lgb.predict(X_val_Sex_F)

In [42]:
#Значения на 5 фолдах для baseline LightGBM для Sex_F
Sex_F_base_lgbm_mean, Sex_F_base_lgbm_std = evaluate_cv_lgb(
    clf_2,
    X_train_Sex_F_,
    y_train_Sex_F_,
    RAND)

Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[54]	valid_0's auc: 0.645477
Fold 1: ROC-AUC:0.6455
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[93]	valid_0's auc: 0.626425
Fold 2: ROC-AUC:0.6264
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.68287
Fold 3: ROC-AUC:0.6829
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[69]	valid_0's auc: 0.68661
Fold 4: ROC-AUC:0.6866
Training until validation scores don't improve for 100 rounds
Training until validation scores don't impr

In [43]:
#Значения на 5 фолдах для optune LightGBM для Sex_F
Sex_F_optune_lgbm_mean, Sex_F_optune_lgbm_std = evaluate_cv_lgb(
    clf_Sex_F_lgb,
    X_train_Sex_F_,
    y_train_Sex_F_,
    RAND)

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[355]	valid_0's auc: 0.678419	valid_0's binary_logloss: 0.594419
Fold 1: ROC-AUC:0.6779
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[390]	valid_0's auc: 0.698184	valid_0's binary_logloss: 0.587055
Fold 2: ROC-AUC:0.6968
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[254]	valid_0's auc: 0.774395	valid_0's binary_logloss: 0.560628
Fold 3: ROC-AUC:0.7744
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[152]	valid_0's auc: 0.70495	valid_0's binary_logloss: 0.602464
Fold 4: ROC-AUC:0.7050
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[131]	valid_0's auc: 0.671315	valid_0's binary_logloss: 0.610561
Fold 5: ROC-AUC:0.6713

Mean ROC-AUC: 0.7051
Std: 0.0368


In [44]:
cv_models_Sex_F.append({'Model_Sex_F':'LightGBM_baseline_Sex_F',
                               'Mean ROC-AUC':Sex_F_base_lgbm_mean,
                               'Std': Sex_F_base_lgbm_std})
cv_models_Sex_F.append({'Model_Sex_F':'LightGBM_optune_Sex_F',
                               'Mean ROC-AUC':Sex_F_optune_lgbm_mean,
                               'Std':Sex_F_optune_lgbm_std})
cv_models_Sex_F_df = pd.DataFrame(cv_models_Sex_F)
round(cv_models_Sex_F_df.set_index('Model_Sex_F'), 3)

,Mean ROC-AUC,Std
Model_Sex_F,,
LogisticRegression_baseline_Sex_F,0.748,0.039
LogisticRegression_optune_Sex_F,0.744,0.030
LightGBM_baseline_Sex_F,0.666,0.025
LightGBM_optune_Sex_F,0.705,0.037


In [45]:
metrics_optuna_Sex_F = pd.concat([metrics_optuna_Sex_F,
    get_metrics_classification(y_test=y_val_Sex_F,
                               y_pred=y_pred_Sex_F_lgb,
                               y_score=clf_Sex_F_lgb.predict_proba(X_val_Sex_F),
                               name='LightGBM_Optuna_Sex_F')])

metrics_optuna_Sex_F

,model,Accuracy,ROC_AUC,Precision,Recall,f1,Logloss
0,LogisticRegression_Optuna_Sex_F,0.642384,0.724895,0.000000,0.000000,0.0,0.616547
0,LightGBM_Optuna_Sex_F,0.701987,0.676212,0.714286,0.277778,0.4,0.613477


In [46]:
metrics_Sex_F

,model,Accuracy,ROC_AUC,Precision,Recall,f1,Logloss
0,LogisticRegression_Baseline_Sex_F,0.708609,0.723813,0.540541,0.425532,0.476190,0.990092
0,LightGBM_Baseline_Sex_F,0.668874,0.723368,0.625000,0.185185,0.285714,0.642981
0,CatBoost_Baseline_Sex_F,0.675497,0.756396,0.777778,0.129630,0.222222,0.572879


study_c_A = optuna.create_study(direction="maximize", study_name="Cat_ADHD")
func = lambda trial: objective_catboost(
    trial, X_train_ADHD_, y_train_ADHD_, N_FOLDS=N_FOLDS, random_state=RAND)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_c_A.optimize(func, n_trials=30, show_progress_bar=True)

study_c_A.best_params

In [47]:
#лучшие параметры для ADHD на CatBoost ROC AUC  0.6794
best_params_CB_ADHD = {'iterations': 927,
 'learning_rate': 0.023655836939172552,
 'depth': 4,
 'l2_leaf_reg': 3.655469435270644,
 'border_count': 86}

In [48]:
clf_ADHD_cb = CatBoostClassifier(**best_params_CB_ADHD)
eval_set = [(X_val_ADHD, y_val_ADHD)]

clf_ADHD_cb.fit(X_train_ADHD_,
        y_train_ADHD_,
        eval_set=eval_set,
        early_stopping_rounds=50,
        use_best_model=True,
        verbose = False
        )

y_pred_ADHD_cb = clf_ADHD_cb.predict(X_val_ADHD)

In [49]:
metrics_optuna_ADHD = pd.concat([metrics_optuna_ADHD,
    get_metrics_classification(y_test=y_val_ADHD,
                               y_pred=y_pred_ADHD_cb,
                               y_score=clf_ADHD_cb.predict_proba(X_val_ADHD),
                               name='CatBoost_Optuna_ADHD')])

round(metrics_optuna_ADHD.set_index('model'), 3)

,Accuracy,ROC_AUC,Precision,Recall,f1,Logloss
model,,,,,,
LogisticRegression_Optuna_ADHD,0.483,0.578,0.721,0.419,0.530,0.739
LightGBM_Optuna_ADHD,0.689,0.733,0.671,0.989,0.800,0.991
LightGBM_Optuna_ADHD,0.689,0.693,0.671,0.989,0.800,0.616
CatBoost_Optuna_ADHD,0.656,0.669,0.650,0.979,0.782,0.624


In [50]:
#Значения на 5 фолдах для baseline CatBoost для ADHD
ADHD_base_cb_mean, ADHD_base_cb = evaluate_cv_catboost(
    clf_c_1,
    X_train_ADHD_,
    y_train_ADHD_,
    RAND)

Fold 1: ROC-AUC:0.7170
Fold 2: ROC-AUC:0.6195
Fold 3: ROC-AUC:0.6278
Fold 4: ROC-AUC:0.7311
Fold 5: ROC-AUC:0.6066

Mean ROC-AUC: 0.6604
Std: 0.0526


In [51]:
#Значения на 5 фолдах для optune CatBoost для ADHD
ADHD_optune_cb_mean, ADHD_optune_cb_std = evaluate_cv_catboost(
    clf_ADHD_cb,
    X_train_ADHD_,
    y_train_ADHD_,
    RAND)

Fold 1: ROC-AUC:0.6970
Fold 2: ROC-AUC:0.6606
Fold 3: ROC-AUC:0.6794
Fold 4: ROC-AUC:0.7443
Fold 5: ROC-AUC:0.6156

Mean ROC-AUC: 0.6794
Std: 0.0423


In [52]:
cv_models_ADHD.append({'Model_ADHD':'CatBoost_baseline_ADHD',
                               'Mean ROC-AUC':ADHD_base_cb_mean,
                               'Std': ADHD_base_cb})
cv_models_ADHD.append({'Model_ADHD':'CatBoost_optune_ADHD',
                               'Mean ROC-AUC':ADHD_optune_cb_mean,
                               'Std':ADHD_optune_cb_std})
cv_models_ADHD_df = pd.DataFrame(cv_models_ADHD)
round(cv_models_ADHD_df.set_index('Model_ADHD'), 3)

,Mean ROC-AUC,Std
Model_ADHD,,
LogisticRegression_baseline_ADHD,0.569,0.032
LogisticRegression_optune_ADHD,0.575,0.028
LightGBM_baseline_ADHD,0.740,0.018
LightGBM_optune_ADHD,0.753,0.041
CatBoost_baseline_ADHD,0.660,0.053
CatBoost_optune_ADHD,0.679,0.042


In [53]:
metrics_ADHD

,model,Accuracy,ROC_AUC,Precision,Recall,f1,Logloss
0,LogisticRegression_Baseline_ADHD,0.635762,0.584679,0.727273,0.761905,0.744186,1.399807
0,LightGBM_eval_Baseline_ADHD,0.701987,0.787970,0.698413,0.926316,0.796380,0.564567
0,CatBoost_eval_Baseline_ADHD,0.688742,0.743233,0.681818,0.947368,0.792952,0.633315


study_c_F = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(),study_name="Cat_SF")
#'pruner = optuna.pruners.NopPruner()'
func = lambda trial: objective_catboost(
    trial, X_train_Sex_F_, y_train_Sex_F_, N_FOLDS=N_FOLDS, random_state=RAND)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_c_F.optimize(func, n_trials=40, show_progress_bar=True)

study_c_F.best_params

In [54]:
#лучшие параметры для Sex_F на CatBoost на 5 фолдах 0.6633
best_params_CB_Sex_F = {'iterations': 748,
 'learning_rate': 0.03813123817525072,
 'depth': 4,
 'l2_leaf_reg': 6.0621261420498325,
 'border_count': 86}

In [55]:
clf_Sex_F_cb = CatBoostClassifier(**best_params_CB_Sex_F)
eval_set = [(X_val_Sex_F, y_val_Sex_F)]

clf_Sex_F_cb.fit(X_train_Sex_F_,
        y_train_Sex_F_,
        eval_set=eval_set,
        verbose=False
        )

y_pred_Sex_F_cb = clf_Sex_F_cb.predict(X_val_Sex_F)

In [56]:
metrics_optuna_Sex_F = pd.concat([metrics_optuna_Sex_F,
    get_metrics_classification(y_test=y_val_Sex_F,
                               y_pred=y_pred_Sex_F_cb,
                               y_score=clf_Sex_F_cb.predict_proba(X_val_Sex_F),
                               name='CatBoost_Optuna_Sex_F')])

metrics_optuna_Sex_F

,model,Accuracy,ROC_AUC,Precision,Recall,f1,Logloss
0,LogisticRegression_Optuna_Sex_F,0.642384,0.724895,0.000000,0.000000,0.000000,0.616547
0,LightGBM_Optuna_Sex_F,0.701987,0.676212,0.714286,0.277778,0.400000,0.613477
0,CatBoost_Optuna_Sex_F,0.662252,0.733104,0.714286,0.092593,0.163934,0.590770


In [57]:
#Значения на 5 фолдах для baseline CatBoost для Sex_F
Sex_F_base_cb_mean, Sex_F_base_cb_std = evaluate_cv_catboost(
    clf_c_2,
    X_train_Sex_F_,
    y_train_Sex_F_,
    RAND)

Fold 1: ROC-AUC:0.6323
Fold 2: ROC-AUC:0.6920
Fold 3: ROC-AUC:0.7635
Fold 4: ROC-AUC:0.6471
Fold 5: ROC-AUC:0.6055

Mean ROC-AUC: 0.6681
Std: 0.0554


In [58]:
#Значения на 5 фолдах для optune CatBoost для Sex_F
Sex_F_optune_cb_mean, Sex_F_optune_cb_std = evaluate_cv_catboost(
    clf_Sex_F_cb,
    X_train_Sex_F_,
    y_train_Sex_F_,
    RAND)

Fold 1: ROC-AUC:0.6344
Fold 2: ROC-AUC:0.6480
Fold 3: ROC-AUC:0.7559
Fold 4: ROC-AUC:0.6806
Fold 5: ROC-AUC:0.5977

Mean ROC-AUC: 0.6633
Std: 0.0534


In [59]:
cv_models_Sex_F.append({'Model_Sex_F':'CatBoost_baseline_Sex_F',
                               'Mean ROC-AUC':Sex_F_base_cb_mean,
                               'Std': Sex_F_base_cb_std})
cv_models_Sex_F.append({'Model_Sex_F':'CatBoost_optune_Sex_F',
                               'Mean ROC-AUC':Sex_F_optune_cb_mean,
                               'Std':Sex_F_optune_cb_std})
cv_models_Sex_F_df = pd.DataFrame(cv_models_Sex_F)
round(cv_models_Sex_F_df.set_index('Model_Sex_F'), 3)

,Mean ROC-AUC,Std
Model_Sex_F,,
LogisticRegression_baseline_Sex_F,0.748,0.039
LogisticRegression_optune_Sex_F,0.744,0.030
LightGBM_baseline_Sex_F,0.666,0.025
LightGBM_optune_Sex_F,0.705,0.037
CatBoost_baseline_Sex_F,0.668,0.055
CatBoost_optune_Sex_F,0.663,0.053


# Стекинг



In [60]:
X_train_ADHD_ = X_train_ADHD_.reset_index(drop=True)
y_train_ADHD_ = y_train_ADHD_.reset_index(drop=True)

oof_lr_1 = pd.Series(np.zeros(len(X_train_ADHD_)))
oof_lgbm_1 = pd.Series(np.zeros(len(X_train_ADHD_)))
oof_cat_1 = pd.Series(np.zeros(len(X_train_ADHD_)))

test_lr_1 = np.zeros(len(X_val_ADHD))
test_lgbm_1 = np.zeros(len(X_val_ADHD))
test_cat_1 = np.zeros(len(X_val_ADHD))

In [61]:
X_train_Sex_F_ = X_train_Sex_F_.reset_index(drop=True)
y_train_Sex_F_ = y_train_Sex_F_.reset_index(drop=True)

oof_lr_2 = np.zeros(len(X_train_Sex_F_))
oof_lgbm_2 = np.zeros(len(X_train_Sex_F_))
oof_cat_2 = np.zeros(len(X_train_Sex_F_))

test_lr_2 = np.zeros(len(X_val_Sex_F))
test_lgbm_2 = np.zeros(len(X_val_Sex_F))
test_cat_2 = np.zeros(len(X_val_Sex_F))

In [62]:
# Pipeline / модели
lr_pipeline_1 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(**best_params_lr_ADHD))
])

lgb_model_1 = LGBMClassifier(**best_params_ADHD_lgb, verbosity = -1)

cat_model_1 = CatBoostClassifier(
    **best_params_CB_ADHD,
    verbose=0
)

# KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf.split(X_train_ADHD_):

    X_tr = X_train_ADHD_.iloc[train_idx]
    X_val = X_train_ADHD_.iloc[val_idx]

    y_tr = y_train_ADHD_.iloc[train_idx]

    #Logistic Regression
    lr_pipeline_1.fit(X_tr, y_tr)
    oof_lr_1[val_idx] = lr_pipeline_1.predict_proba(X_val)[:, 1]

    #LightGBM
    lgb_model_1.fit(X_tr, y_tr)
    oof_lgbm_1[val_idx] = lgb_model_1.predict_proba(X_val)[:, 1]

    #CatBoost
    cat_model_1.fit(X_tr, y_tr)
    oof_cat_1[val_idx] = cat_model_1.predict_proba(X_val)[:, 1]

In [63]:
print("LR ROC-AUC:", roc_auc_score(y_train_ADHD_, oof_lr_1))
print("LGBM ROC-AUC:", roc_auc_score(y_train_ADHD_, oof_lgbm_1))
print("CatBoost ROC-AUC:", roc_auc_score(y_train_ADHD_, oof_cat_1))

LR ROC-AUC: 0.5650484849859043
LGBM ROC-AUC: 0.6596981308905667
CatBoost ROC-AUC: 0.6726224217551338


In [64]:
# OOF массивы для train 
X_meta_train_1 = np.column_stack([oof_lr_1, oof_lgbm_1, oof_cat_1])
y_meta_train_1 = y_train_ADHD_.values  # длина = len(X_meta_train)

y_meta_val_1 = y_train_ADHD_.values  # длина = len(X_meta_val)

In [65]:
len(X_train_ADHD)

941

In [66]:
meta_model_1 = LogisticRegression()
meta_model_1.fit(X_meta_train_1, y_train_ADHD_)

LogisticRegression()

In [67]:
lr_pipeline_1.fit(X_train_ADHD_, y_train_ADHD_)
lgb_model_1.fit(X_train_ADHD_, y_train_ADHD_)
cat_model_1.fit(X_train_ADHD_, y_train_ADHD_)

In [68]:
X_meta_val_1 = np.column_stack([
    lr_pipeline_1.predict_proba(X_val_ADHD)[:, 1],
    lgb_model_1.predict_proba(X_val_ADHD)[:, 1],
    cat_model_1.predict_proba(X_val_ADHD)[:, 1]
])

In [69]:
X_meta_val_1.shape[0] == len(y_val_ADHD)

True

In [70]:
oof_meta_ADHD = meta_model_1.predict_proba(X_meta_train_1)[:, 1]

In [71]:
print("Stacking ROC-AUC:", roc_auc_score(y_train_ADHD_, oof_meta_ADHD))

Stacking ROC-AUC: 0.6760688993190184


In [72]:
# Pipeline / модели
lr_pipeline_2 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

lgb_model_2 = LGBMClassifier(**best_params_Sex_F_lgbm, verbosity = -1)

cat_model_2 = CatBoostClassifier()

# KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf.split(X_train_Sex_F_):

    X_tr = X_train_Sex_F_.iloc[train_idx]
    X_val = X_train_Sex_F_.iloc[val_idx]

    y_tr = y_train_Sex_F_.iloc[train_idx]

    #Logistic Regression
    lr_pipeline_2.fit(X_tr, y_tr)
    oof_lr_2[val_idx] = lr_pipeline_2.predict_proba(X_val)[:, 1]

    #LightGBM
    lgb_model_2.fit(X_tr, y_tr)
    oof_lgbm_2[val_idx] = lgb_model_2.predict_proba(X_val)[:, 1]

    #CatBoost
    cat_model_2.fit(X_tr, y_tr)
    oof_cat_2[val_idx] = cat_model_2.predict_proba(X_val)[:, 1]

Learning rate set to 0.008469
0:	learn: 0.6908085	total: 2.26s	remaining: 37m 36s
1:	learn: 0.6886030	total: 4.26s	remaining: 35m 25s
2:	learn: 0.6867400	total: 6.21s	remaining: 34m 24s
3:	learn: 0.6849954	total: 8.13s	remaining: 33m 44s
4:	learn: 0.6828964	total: 10.1s	remaining: 33m 35s
5:	learn: 0.6813312	total: 12.1s	remaining: 33m 28s
6:	learn: 0.6794630	total: 14.1s	remaining: 33m 23s
7:	learn: 0.6772889	total: 16.1s	remaining: 33m 13s
8:	learn: 0.6754546	total: 18s	remaining: 32m 57s
9:	learn: 0.6734036	total: 19.9s	remaining: 32m 51s
10:	learn: 0.6714266	total: 21.9s	remaining: 32m 45s
11:	learn: 0.6691441	total: 23.8s	remaining: 32m 38s
12:	learn: 0.6674061	total: 25.7s	remaining: 32m 32s
13:	learn: 0.6654216	total: 27.7s	remaining: 32m 31s
14:	learn: 0.6636903	total: 29.7s	remaining: 32m 32s
15:	learn: 0.6616958	total: 31.8s	remaining: 32m 35s
16:	learn: 0.6599812	total: 33.9s	remaining: 32m 41s
17:	learn: 0.6581721	total: 35.9s	remaining: 32m 40s
18:	learn: 0.6557816	total: 

In [73]:
print("LR ROC-AUC:", roc_auc_score(y_train_Sex_F_, oof_lr_2))
print("LGBM ROC-AUC:", roc_auc_score(y_train_Sex_F_, oof_lgbm_2))
print("CatBoost ROC-AUC:", roc_auc_score(y_train_Sex_F_, oof_cat_2))

LR ROC-AUC: 0.7741967593086335
LGBM ROC-AUC: 0.7045339817560025
CatBoost ROC-AUC: 0.6756322476519564


In [74]:
# --- 1. OOF массивы для train (которые уже есть) ---
X_meta_train_2 = np.column_stack([oof_lr_2, oof_lgbm_2, oof_cat_2])
y_meta_train_2 = y_train_Sex_F_.values  # длина = len(X_meta_train)

y_meta_val_2 = y_train_Sex_F_.values  # длина = len(X_meta_val)

In [75]:
meta_model_2 = LogisticRegression()
meta_model_2.fit(X_meta_train_2, y_train_Sex_F_)

LogisticRegression()

In [76]:
lr_pipeline_2.fit(X_train_Sex_F_, y_train_Sex_F_)
lgb_model_2.fit(X_train_Sex_F_, y_train_Sex_F_)
cat_model_2.fit(X_train_Sex_F_, y_train_Sex_F_)

Learning rate set to 0.009316
0:	learn: 0.6909742	total: 2.33s	remaining: 38m 44s
1:	learn: 0.6885555	total: 4.33s	remaining: 36m
2:	learn: 0.6867613	total: 6.26s	remaining: 34m 41s
3:	learn: 0.6848765	total: 8.23s	remaining: 34m 10s
4:	learn: 0.6831189	total: 10.2s	remaining: 33m 50s
5:	learn: 0.6812263	total: 12.2s	remaining: 33m 35s
6:	learn: 0.6791170	total: 14.1s	remaining: 33m 25s
7:	learn: 0.6776160	total: 16s	remaining: 33m 9s
8:	learn: 0.6755188	total: 18s	remaining: 32m 59s
9:	learn: 0.6729870	total: 20s	remaining: 32m 56s
10:	learn: 0.6709572	total: 21.9s	remaining: 32m 53s
11:	learn: 0.6688320	total: 23.9s	remaining: 32m 45s
12:	learn: 0.6665931	total: 25.8s	remaining: 32m 39s
13:	learn: 0.6648568	total: 27.8s	remaining: 32m 38s
14:	learn: 0.6629037	total: 29.8s	remaining: 32m 39s
15:	learn: 0.6609991	total: 32s	remaining: 32m 46s
16:	learn: 0.6591638	total: 34.1s	remaining: 32m 52s
17:	learn: 0.6572115	total: 36.2s	remaining: 32m 54s
18:	learn: 0.6553705	total: 38.2s	remai

In [77]:
X_meta_val_2 = np.column_stack([
    lr_pipeline_2.predict_proba(X_val_Sex_F)[:, 1],
    lgb_model_2.predict_proba(X_val_Sex_F)[:, 1],
    cat_model_2.predict_proba(X_val_Sex_F)[:, 1]
])

In [78]:
oof_meta_Sex_F = meta_model_2.predict_proba(X_meta_train_2)[:, 1]

In [79]:
print("Stacking ROC-AUC:", roc_auc_score(y_train_Sex_F_, oof_meta_Sex_F))

Stacking ROC-AUC: 0.759379732525649


In [80]:
metrics_meta_ADHD = get_metrics_classification(y_test=y_val_ADHD,
                               y_pred=meta_model_1.predict(X_meta_val_1),
                               y_score=meta_model_1.predict_proba(X_meta_val_1),
                               name='Meta_LR_ADHD')

metrics_meta_ADHD

,model,Accuracy,ROC_AUC,Precision,Recall,f1,Logloss
0,Meta_LR_ADHD,0.688742,0.697556,0.6875,0.926316,0.789238,0.597453


In [81]:
metrics_meta_Sex_F = get_metrics_classification(y_test=y_val_Sex_F,
                               y_pred=meta_model_2.predict(X_meta_val_2),
                               y_score=meta_model_2.predict_proba(X_meta_val_2),
                               name='Meta_LR_Sex_F')

metrics_meta_Sex_F

,model,Accuracy,ROC_AUC,Precision,Recall,f1,Logloss
0,Meta_LR_Sex_F,0.741722,0.779878,0.702703,0.481481,0.571429,0.546335


In [82]:
Full_metrics_ADHD = pd.concat([metrics_ADHD, metrics_optuna_ADHD, metrics_meta_ADHD], axis = 0, ignore_index = True)

round(Full_metrics_ADHD.set_index('model'), 3)

,Accuracy,ROC_AUC,Precision,Recall,f1,Logloss
model,,,,,,
LogisticRegression_Baseline_ADHD,0.636,0.585,0.727,0.762,0.744,1.400
LightGBM_eval_Baseline_ADHD,0.702,0.788,0.698,0.926,0.796,0.565
CatBoost_eval_Baseline_ADHD,0.689,0.743,0.682,0.947,0.793,0.633
LogisticRegression_Optuna_ADHD,0.483,0.578,0.721,0.419,0.530,0.739
LightGBM_Optuna_ADHD,0.689,0.733,0.671,0.989,0.800,0.991
LightGBM_Optuna_ADHD,0.689,0.693,0.671,0.989,0.800,0.616
CatBoost_Optuna_ADHD,0.656,0.669,0.650,0.979,0.782,0.624
Meta_LR_ADHD,0.689,0.698,0.688,0.926,0.789,0.597


In [83]:
Full_metrics_Sex_F = pd.concat([metrics_Sex_F, metrics_optuna_Sex_F, metrics_meta_Sex_F], axis = 0, ignore_index = True)

round(Full_metrics_Sex_F.set_index('model'), 4)

,Accuracy,ROC_AUC,Precision,Recall,f1,Logloss
model,,,,,,
LogisticRegression_Baseline_Sex_F,0.7086,0.7238,0.5405,0.4255,0.4762,0.9901
LightGBM_Baseline_Sex_F,0.6689,0.7234,0.6250,0.1852,0.2857,0.6430
CatBoost_Baseline_Sex_F,0.6755,0.7564,0.7778,0.1296,0.2222,0.5729
LogisticRegression_Optuna_Sex_F,0.6424,0.7249,0.0000,0.0000,0.0000,0.6165
LightGBM_Optuna_Sex_F,0.7020,0.6762,0.7143,0.2778,0.4000,0.6135
CatBoost_Optuna_Sex_F,0.6623,0.7331,0.7143,0.0926,0.1639,0.5908
Meta_LR_Sex_F,0.7417,0.7799,0.7027,0.4815,0.5714,0.5463


In [86]:
from sklearn.base import clone

In [92]:
def evaluate_stacking_cv(models,
                        X,
                        y,
                        n_folds=5,
                        inner_folds=5,
                        random_state=10):
    
    outer_cv = StratifiedKFold(n_splits=n_folds,
                                shuffle=True,
                                random_state=random_state)

    scores = []

    for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X, y), 1):

        print(f"\nFold {fold}")

        # =========================
        # OUTER TRAIN / VALIDATION
        # =========================

        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        # =========================
        # INNER CV
        # Получаем OOF predictions
        # для meta-model
        # =========================

        inner_cv = StratifiedKFold(n_splits=inner_folds,
                                   shuffle=True,
                                   random_state=random_state)

        oof_train = np.zeros((len(X_train), len(models)))

        for inner_train_idx, inner_val_idx in inner_cv.split(X_train, y_train):

            X_inner_train = X_train.iloc[inner_train_idx]
            X_inner_val = X_train.iloc[inner_val_idx]

            y_inner_train = y_train.iloc[inner_train_idx]
            y_inner_val = y_train.iloc[inner_val_idx]

            for model_idx, model in enumerate(models):

                model_inner = clone(model)

                model_inner.fit(X_inner_train, y_inner_train)

                oof_train[inner_val_idx, model_idx] = model_inner.predict_proba(X_inner_val)[:, 1]

        # =========================
        # META-MODEL
        # =========================

        meta_model = LogisticRegression(max_iter=1000)

        meta_model.fit(oof_train, y_train)

        # =========================
        # ОБУЧАЕМ BASE MODELS
        # НА ВСЁМ OUTER TRAIN
        # =========================

        meta_val = np.zeros((len(X_val), len(models)))

        for model_idx, model in enumerate(models):

            model_outer = clone(model)

            model_outer.fit(X_train, y_train)

            meta_val[:,model_idx] = model_outer.predict_proba(X_val)[:, 1]

        # =========================
        # FINAL STACKING PREDICTION
        # =========================

        y_pred_proba = meta_model.predict_proba(meta_val)[:, 1]

        score = roc_auc_score(y_val, y_pred_proba)

        scores.append(score)

        print(f"Fold {fold}: ROC-AUC = {score:.4f}")

    # =========================
    # FINAL RESULTS
    # =========================

    mean_score = np.mean(scores)
    std_score = np.std(scores)

    print(f"\nMean ROC-AUC: {mean_score:.4f}")

    print(f"Std: {std_score:.4f}")

    return mean_score, std_score

In [87]:
models_ADHD = [lr_pipeline_1,
               lgb_model_1,
               cat_model_1]

In [93]:
stack_mean_ADHD, stack_std_ADHD = evaluate_stacking_cv(
    models=models_ADHD,
    X=X_train_ADHD_,
    y=y_train_ADHD_,
    n_folds=5,
    inner_folds=5,
    random_state=10
)


Fold 1
Fold 1: ROC-AUC = 0.6595

Fold 2
Fold 2: ROC-AUC = 0.6678

Fold 3
Fold 3: ROC-AUC = 0.7165

Fold 4
Fold 4: ROC-AUC = 0.7470

Fold 5
Fold 5: ROC-AUC = 0.6119

Mean ROC-AUC: 0.6805
Std: 0.0470


In [88]:
models_Sex_F = [lr_pipeline_2,
                lgb_model_2,
                cat_model_2]

In [94]:
stack_mean_Sex_F, stack_std_Sex_F = evaluate_stacking_cv(
    models=models_Sex_F,
    X=X_train_Sex_F_,
    y=y_train_Sex_F_,
    n_folds=5,
    inner_folds=5,
    random_state=10
)


Fold 1
Learning rate set to 0.007696
0:	learn: 0.6908162	total: 2.89s	remaining: 48m 10s
1:	learn: 0.6890553	total: 4.95s	remaining: 41m 9s
2:	learn: 0.6877868	total: 7.05s	remaining: 39m 2s
3:	learn: 0.6856873	total: 9.05s	remaining: 37m 33s
4:	learn: 0.6835897	total: 11.1s	remaining: 36m 45s
5:	learn: 0.6821185	total: 13.1s	remaining: 36m 11s
6:	learn: 0.6803981	total: 15.1s	remaining: 35m 41s
7:	learn: 0.6783302	total: 17.1s	remaining: 35m 21s
8:	learn: 0.6763381	total: 19.1s	remaining: 35m 5s
9:	learn: 0.6746345	total: 21.1s	remaining: 34m 51s
10:	learn: 0.6725994	total: 23.1s	remaining: 34m 40s
11:	learn: 0.6709938	total: 25.1s	remaining: 34m 25s
12:	learn: 0.6690600	total: 27.1s	remaining: 34m 17s
13:	learn: 0.6675179	total: 29.2s	remaining: 34m 15s
14:	learn: 0.6658758	total: 31.3s	remaining: 34m 15s
15:	learn: 0.6637911	total: 33.5s	remaining: 34m 19s
16:	learn: 0.6627425	total: 35.5s	remaining: 34m 15s
17:	learn: 0.6609584	total: 37.8s	remaining: 34m 20s
18:	learn: 0.6592822	

In [95]:
cv_models_ADHD.append({'Model_ADHD':'Meta_ADHD',
                               'Mean ROC-AUC':stack_mean_ADHD,
                               'Std':stack_std_ADHD})
cv_models_ADHD_df = pd.DataFrame(cv_models_ADHD)
round(cv_models_ADHD_df.set_index('Model_ADHD'), 3)

,Mean ROC-AUC,Std
Model_ADHD,,
LogisticRegression_baseline_ADHD,0.569,0.032
LogisticRegression_optune_ADHD,0.575,0.028
LightGBM_baseline_ADHD,0.740,0.018
LightGBM_optune_ADHD,0.753,0.041
CatBoost_baseline_ADHD,0.660,0.053
CatBoost_optune_ADHD,0.679,0.042
Meta_ADHD,0.681,0.047


In [96]:
cv_models_Sex_F.append({'Model_Sex_F':'Meta_Sex_F',
                               'Mean ROC-AUC':stack_mean_Sex_F,
                               'Std':stack_std_Sex_F})
cv_models_Sex_F_df = pd.DataFrame(cv_models_Sex_F)
round(cv_models_Sex_F_df.set_index('Model_Sex_F'), 3)

,Mean ROC-AUC,Std
Model_Sex_F,,
LogisticRegression_baseline_Sex_F,0.748,0.039
LogisticRegression_optune_Sex_F,0.744,0.030
LightGBM_baseline_Sex_F,0.666,0.025
LightGBM_optune_Sex_F,0.705,0.037
CatBoost_baseline_Sex_F,0.668,0.055
CatBoost_optune_Sex_F,0.663,0.053
Meta_Sex_F,0.751,0.030


# Финальные модели

In [98]:
X_meta_test_2 = np.column_stack([
    lr_pipeline_2.predict_proba(X_test_Sex_F)[:, 1],
    lgb_model_2.predict_proba(X_test_Sex_F)[:, 1],
    cat_model_2.predict_proba(X_test_Sex_F)[:, 1]
])

In [99]:
metrics_final = get_metrics_classification(y_test=y_test_ADHD,
                               y_pred=clf_ADHD_lgb.predict(X_test_ADHD),
                               y_score=clf_ADHD_lgb.predict_proba(X_test_ADHD),
                               name='LightGBM_optune_ADHD')
metrics_final = pd.concat([metrics_final,
    get_metrics_classification(y_test=y_test_Sex_F,
                               y_pred=meta_model_2.predict(X_meta_test_2),
                               y_score=meta_model_2.predict_proba(X_meta_test_2),
                               name='Meta_Sex_F')])
round(metrics_final.set_index('model'), 3)

,Accuracy,ROC_AUC,Precision,Recall,f1,Logloss
model,,,,,,
LightGBM_optune_ADHD,0.746,0.773,0.746,0.964,0.841,0.522
Meta_Sex_F,0.716,0.755,0.579,0.434,0.496,0.553
